## **00 Build Data**
---

Here we explore the raw data, and do pre-processing to build the datasets the models will be using.

In [ ]:
# Imports:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import sys
import os

In [ ]:
# This lets you change whether the notebook should run for Norway, US, or both countries,
# a pattern implemented to avoid re-running code unnecessarily - particularly for model training.
# To update, modify COUNTRY in CONFIG.py

# Add the parent directory to Python's search path so it can find CONFIG.py
sys.path.append(os.path.abspath('..'))

from CONFIG import COUNTRY
countries = ["us", "norway"] if COUNTRY == "both" else [COUNTRY] 

In [ ]:
# Plotting parameters
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'font.family': 'serif',
})
sns.set_style('whitegrid')

## Yield Curve Data

### US Data - Liu-Wu dataset
We read the dataset and select the maturities that match the Norwegian dataset. Norges Bank publishes Zero Coupon Yield data for the following maturities:

* 6 months
* 9 months
* 1 year
* 2 years
* 3 years
* 4 years
* 5 years
* 6 years
* 7 years
* 8 years
* 9 years
* 10 years 



In [ ]:
# Load Liu-Wu and US data
lw_raw = pd.read_excel('../data/raw/us/LW_monthly.xlsx', header=0, skiprows=8)
lw_raw.head()

,Unnamed: 0,1 m,2 m,3 m,4 m,5 m,6 m,7 m,8 m,9 m,...,351 m,352 m,353 m,354 m,355 m,356 m,357 m,358 m,359 m,360 m
0,196106,2.092599,2.193762,2.289332,2.380383,2.468033,2.552678,2.633305,2.707428,2.771769,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,196107,1.798512,1.965591,2.111735,2.240318,2.354286,2.455535,2.544454,2.620127,2.681560,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,196108,2.022547,2.163381,2.297206,2.424001,2.542289,2.649290,2.741553,2.816288,2.873426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,196109,1.976399,2.130537,2.277331,2.414313,2.536931,2.638611,2.713540,2.762042,2.792608,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,196110,1.981141,2.129609,2.272949,2.407495,2.528128,2.628622,2.704765,2.758808,2.798653,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Norges Bank maturities:
nb_maturities = [
    '6 m',    # 6 months
    '9 m',    # 9 months
    '12 m',   # 1 year
    '24 m',   # 2 years
    '36 m',   # 3 years
    '48 m',   # 4 years
    '60 m',   # 5 years
    '72 m',   # 6 years
    '84 m',   # 7 years
    '96 m',   # 8 years
    '108 m',  # 9 years
    '120 m'   # 10 years
]

# Remove leading or trailing spaces from all column names
lw_raw.columns = lw_raw.columns.str.strip()

# Extract dates + nb maturities
lw_subset = lw_raw[['Unnamed: 0'] + nb_maturities].copy()

# Rename columns:
lw_subset.rename(columns={
    'Unnamed: 0': 'Date',
    '6 m': '6m',
    '9 m': '9m',
    '12 m': '1y',
    '24 m': '2y',
    '36 m': '3y',
    '48 m': '4y',
    '60 m': '5y',
    '72 m': '6y',
    '84 m': '7y',
    '96 m': '8y',
    '108 m': '9y',
    '120 m': '10y'
}, inplace=True)
# lw_subset.head()

A bunch of NaNs for the 8, 9 and 10 year maturities, but we are only using data from 2003 due to the limited availability of Norwegian data:

In [ ]:
# Keep only data from January 2003 and out
lw_subset = lw_subset[lw_subset['Date'] >= 200301]

lw_subset = lw_subset.reset_index(drop=True)
#lw_subset.describe()

In [ ]:
lw_subset.tail()

,Date,6m,9m,1y,2y,3y,4y,5y,6y,7y,8y,9y,10y
259,202408,4.726013,4.504532,4.329078,3.877377,3.733707,3.697840,3.695434,3.730766,3.769364,3.813019,3.851071,3.863879
260,202409,4.295713,4.094602,3.936461,3.611258,3.526146,3.526613,3.542847,3.588507,3.638934,3.692485,3.736240,3.753135
261,202410,4.389511,4.319004,4.261855,4.106225,4.085483,4.106356,4.118602,4.153857,4.192476,4.212841,4.230589,4.232085
262,202411,4.321200,4.301971,4.273649,4.124058,4.051641,4.038893,4.030609,4.049707,4.080090,4.105403,4.125853,4.125693
263,202412,4.170671,4.172976,4.174661,4.199344,4.240230,4.307826,4.348454,4.406289,4.456465,4.490777,4.525956,4.537210


### Nelson Siegel Factors

In [ ]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Helper functions for Nelson Siegel
# ---------------------------------------------------------------------------

def ns_loadings(maturities: np.ndarray, lam: float = 0.0609) -> np.ndarray:
    """
    Computes the Nelson-Siegel factor loadings (the "shapes") for a vector of maturities.
    
    The Nelson-Siegel model decomposes the yield curve into three latent factors:
      1. Level:     Long-term component (loading = 1.0 for all maturities)
      2. Slope:     Short-term component (starts high, decays to 0)
      3. Curvature: Medium-term component (starts at 0, humps in the middle, decays to 0)
      
    Parameters:
    - maturities: Array of bond maturities (in months).
    - lam (lambda): Decay parameter determining where the "hump" for curvature occurs. 
                    0.0609 is the standard Diebold-Li (2006) formulation.
    """
    tau = np.asarray(maturities, dtype=float)
    lt = lam * tau

    level = np.ones_like(tau)
    slope = (1 - np.exp(-lt)) / lt
    curvature = slope - np.exp(-lt)

    return np.column_stack([level, slope, curvature])


def extract_ns_factors(
    yields_matrix: np.ndarray,
    maturities: np.ndarray,
    lam: float = 0.0609,
    dates: pd.DatetimeIndex | None = None,
) -> pd.DataFrame:
    """
    Extracts Nelson-Siegel factors via cross-sectional OLS for each date,
    returning a DataFrame with factors, RMSE, and observation count.

    Parameters:
    - yields_matrix: (T x N) array of yields, rows = dates, cols = maturities.
    - maturities:    (N,) array of bond maturities (in months).
    - lam:           NS decay parameter (default 0.0609, Diebold-Li 2006).
    - dates:         Optional DatetimeIndex used as the DataFrame index.

    Returns:
    - DataFrame with columns [level, slope, curvature, rmse, n_obs],
      indexed by date if provided.
    """
    T = yields_matrix.shape[0]

    factors = np.empty((T, 3))
    rmses = np.empty(T)
    n_obs = np.empty(T, dtype=int)

    X_full = ns_loadings(maturities, lam)

    for t in range(T):
        y = yields_matrix[t]
        mask = ~np.isnan(y)
        n_valid = mask.sum()

        if n_valid < 3:
            factors[t] = np.nan
            rmses[t] = np.nan
            n_obs[t] = n_valid
            continue

        if mask.all():
            X = X_full
            y_clean = y
        else:
            X = ns_loadings(maturities[mask], lam)
            y_clean = y[mask]

        beta, _, _, _ = np.linalg.lstsq(X, y_clean, rcond=None)

        factors[t] = beta
        fitted = X @ beta
        rmses[t] = np.sqrt(np.mean((y_clean - fitted) ** 2))
        n_obs[t] = n_valid

    df = pd.DataFrame(
        factors,
        columns=["level", "slope", "curvature"],
        index=dates,
    )
    df["rmse"] = rmses
    df["n_obs"] = n_obs

    return df

#### NS for Liu-Wu data
We now have 264 observations for all maturities, spanning January 2003 to December 2024. We can proceed with finding the Nelson Siegel factors.

In [ ]:
# The NB maturities in months
maturities_months = np.array([6, 9, 12, 24, 36, 48, 60, 72, 84, 96, 108, 120])

# Extract yield columns as a numpy array
yields_array = lw_subset.drop(columns=["Date"]).values

# Parse dates
dates = pd.to_datetime(lw_subset["Date"].astype(str), format="%Y%m")
# Note that it automatically assigns 1st of the month - not really of consequence

# Extract NS factors
lw_ns = extract_ns_factors(
    yields_matrix=yields_array,
    maturities=maturities_months,
    dates=dates,
)

lw_ns.describe()

,level,slope,curvature,rmse,n_obs
count,264.000000,264.000000,264.000000,264.000000,264.0
mean,3.697842,-1.820219,-3.041449,0.049150,12.0
std,1.288933,1.880784,2.718161,0.035080,0.0
min,0.769744,-5.297363,-10.187321,0.007283,12.0
25%,2.752680,-3.279994,-5.449192,0.024895,12.0
50%,3.749719,-1.907079,-2.489292,0.039328,12.0
75%,4.877099,-0.447071,-1.091457,0.060160,12.0
max,6.615353,2.581539,4.667074,0.215538,12.0


Great. We now save the preprocessed data and NS data:

In [ ]:
# Save the cleaned Liu-Wu data with selected maturities & filtered dates
lw_subset.to_csv("../data/processed/us/us_yields_cleaned.csv", index=False)
print(f"Saved cleaned US yields to data/")


# Save Liu-Wu NS factors
ns_df.to_csv("../data/processed/us/us_ns_factors.csv", index=False)
print(f"Saved US NS factors to data/")

Saved cleaned yields to data
Saved NS factors to data


## Norwegian Datasets

In [ ]:
# Load Norges Bank data
nb_raw = pd.read_excel('../data/raw/norway/GOVT_ZEROCOUPON.xlsx', header=0, skiprows=8)
#nb_raw.head()

c:\Users\adne_\anaconda3\envs\torch-gpu\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
# Load Morten, Petter and Sjur data
NB1MR_raw = pd.read_excel('../data/raw/norway/NB1MRcompiled.xlsx', header=0, skiprows=8)
#NB1MR_raw.head()